[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/09_sandboxing.ipynb)

# 09. Sandboxing

**Part 2: Tools** · about 25 minutes

Notebook 08 ended on an uncomfortable fact. Permissions read the *text* of a command, and text doesn't tell you what code will do once it runs. `python -m unittest` looks harmless, and it's allowed, but it runs whatever the test files say, and the model may have written those.

A **sandbox** works from the other side. It doesn't care what the command says. It limits what the running process is *able* to do, so even code nobody has read can only do limited damage.

There isn't one sandbox. There are layers, each stopping different things:

| Layer | Stops | Doesn't stop |
|---|---|---|
| Workspace folder and minimal environment (notebooks 05, 07) | Accidental edits elsewhere, leaked API keys | Anything done on purpose |
| Process-group timeout (notebook 07) | Commands that never end | Fast damage |
| **Resource limits** (this notebook) | CPU burners, disk fillers, memory hogs | Reading your files, using the network |
| **Containers** (this notebook) | Reading files outside the project, network access | Kernel exploits, which are rare but real |

This notebook builds the middle two layers and shows exactly what each one leaves open, because a sandbox you overestimate is worse than one you understand.

**By the end you will have:**

- killed a runaway process by CPU time, and stopped a disk-filling write
- seen with your own eyes what resource limits *don't* stop
- the container version, run when Docker is available
- a sandboxed `run_command` for the Part 2 agent

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import os
import re
import shlex
import shutil
import signal
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

workspace = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()
PYTHON = shlex.quote(sys.executable)

HAS_LIMITS = os.name == "posix"  # resource limits are a Unix feature
if HAS_LIMITS:
    import resource
print("Resource limits available:", HAS_LIMITS)

Resource limits available: True


## 1. Resource limits

Unix lets a process set ceilings that the kernel enforces on it and on every process it starts. We set them in `preexec_fn`, which runs in the child process right before the command starts:

- **CPU time** (`RLIMIT_CPU`): after N seconds of actual computing, the kernel sends `SIGXCPU`, which ends the process. This is different from a wall-clock timeout, because a process that's asleep or waiting uses no CPU.
- **File size** (`RLIMIT_FSIZE`): no single file can grow past the limit.
- **Memory** (`RLIMIT_AS`): a cap on how much memory the process can reserve. This one works reliably only on Linux. macOS mostly ignores it.

In [3]:
def make_limits(cpu_seconds=5, file_mb=10, memory_mb=512):
    """Return a function that applies the limits inside the child process."""
    def apply():
        resource.setrlimit(resource.RLIMIT_CPU, (cpu_seconds, cpu_seconds + 1))
        size = file_mb * 1024 * 1024
        resource.setrlimit(resource.RLIMIT_FSIZE, (size, size))
        if sys.platform.startswith("linux"):
            memory = memory_mb * 1024 * 1024
            resource.setrlimit(resource.RLIMIT_AS, (memory, memory))
    return apply


def describe(code):
    """Explain an exit code, including deaths by signal, in words a model can act on."""
    if code is None:
        return "stopped: took longer than the time limit"
    if code < 0 or code > 128:  # negative when Python started it directly, 128+N when a shell did
        number = -code if code < 0 else code - 128
        try:
            name = signal.Signals(number).name
        except ValueError:
            return f"exit code: {code}"
        why = {"SIGXCPU": "used more CPU time than allowed", "SIGXFSZ": "wrote a file larger than allowed",
               "SIGKILL": "was killed", "SIGSEGV": "crashed"}.get(name, "was stopped by a signal")
        return f"stopped: {why} ({name})"
    return f"exit code: {code}"


def run_limited(command, timeout=30, **limits):
    options = dict(shell=True, cwd=workspace, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                   env={**{k: os.environ[k] for k in ("PATH", "SYSTEMROOT") if k in os.environ},
                        "HOME": str(workspace)})
    if HAS_LIMITS:
        options.update(start_new_session=True, preexec_fn=make_limits(**limits))
    started = time.perf_counter()
    process = subprocess.Popen(command, **options)
    try:
        output, _ = process.communicate(timeout=timeout)
        code = process.returncode
    except subprocess.TimeoutExpired:
        if HAS_LIMITS:
            os.killpg(process.pid, signal.SIGKILL)
        else:
            subprocess.run(["taskkill", "/T", "/F", "/PID", str(process.pid)], capture_output=True)
        output, _ = process.communicate()
        code = None
    return code, output.replace(str(workspace), "."), time.perf_counter() - started

### A process that never stops computing

An infinite loop, with a generous 30-second wall clock but only 2 seconds of CPU allowed.

In [4]:
if HAS_LIMITS:
    code, output, seconds = run_limited(f'{PYTHON} -c "while True: pass"', timeout=30, cpu_seconds=2)
    print(f"{describe(code)} after {seconds:.1f}s")
    assert "SIGXCPU" in describe(code)
    print("Checkpoint passed.")
else:
    print("Skipped: this system doesn't support resource limits. On Windows, only the wall-clock timeout protects you.")

stopped: used more CPU time than allowed (SIGXCPU) after 2.0s
Checkpoint passed.


### A process that tries to fill the disk

A 50 MB write against a 10 MB limit. Python turns the kernel's refusal into an ordinary exception, so the process gets a clear error instead of a silent full disk.

In [5]:
if HAS_LIMITS:
    code, output, _ = run_limited(f"{PYTHON} -c \"open('big.bin', 'wb').write(b'x' * 50_000_000)\"", file_mb=10)
    size_mb = (workspace / "big.bin").stat().st_size / 1024 / 1024
    print(describe(code))
    print(output.strip().splitlines()[-1])
    print(f"big.bin stopped growing at {size_mb:.0f} MB")
    (workspace / "big.bin").unlink()
    assert size_mb <= 10
    print("Checkpoint passed.")
else:
    print("Skipped: no resource limits on this system.")

exit code: 1
OSError: [Errno 27] File too large
big.bin stopped growing at 10 MB
Checkpoint passed.


### A process that grabs all the memory (Linux only)

In [6]:
if HAS_LIMITS and sys.platform.startswith("linux"):
    code, output, _ = run_limited(f'{PYTHON} -c "x = bytearray(2_000_000_000)"', memory_mb=512)
    print(describe(code))
    print(output.strip().splitlines()[-1])
    assert "MemoryError" in output
    print("Checkpoint passed.")
else:
    print("Skipped: memory limits are only reliable on Linux. Try this notebook on Colab to see it.")

Skipped: memory limits are only reliable on Linux. Try this notebook on Colab to see it.


## 2. What resource limits don't stop

Now for the honest part. A process with every limit above can still:

1. **Read any file your user account can read.** Your SSH keys, your browser profile, other projects.
2. **Use the network,** for example to send whatever it read somewhere.

Let's prove the first one with a pretend secret outside the workspace. (We print only what was read, not where the file lives.)

In [7]:
elsewhere = Path(tempfile.mkdtemp(prefix="not-the-workspace-")).resolve()
(elsewhere / "secret.txt").write_text("TOP SECRET: the launch is cancelled")

code, output, _ = run_limited(f"cat {shlex.quote(str(elsewhere / 'secret.txt'))}", cpu_seconds=2)
print("A limited process outside the workspace read:", output.strip().replace(str(elsewhere), "..."))

A limited process outside the workspace read: TOP SECRET: the launch is cancelled


It read it without any trouble. Limits on CPU, disk and memory say nothing about *which* files a process may touch.

The network is the same story. Any process that could open the file above could just as easily send it somewhere. We won't make a network call here, but nothing in this layer would stop one.

To close these gaps you need the operating system to give the process its **own view of the world**: a filesystem that contains only the project, and no network at all. That's what containers do.

## 3. Containers

A container runs a process with its own filesystem and, if you ask, no network. Docker is the common way to get one. Every flag below closes a specific gap:

| Flag | Effect |
|---|---|
| `--network none` | No network at all |
| `-v <workspace>:/work` | Only the project folder is visible. Your home folder doesn't exist in there. |
| `--read-only` | Everything outside `/work` is read-only |
| `--memory 256m --cpus 1` | Memory and CPU caps that work on every OS |
| `--pids-limit 64` | No fork bombs |
| `--user 1000:1000` | Not root inside the container |
| `--cap-drop ALL` and `no-new-privileges` | Removes the kernel's special privileges |

The next cell runs the same two "escapes" from section 2 inside a container, if Docker is available. On Colab and many laptops it isn't, so you'll see the command instead of the results. The automated tests for this repo run on Linux with Docker, so this cell does get exercised.

In [8]:
IMAGE = "python:3.12-slim"


def docker_ready():
    if not shutil.which("docker"):
        return False
    return subprocess.run(["docker", "info"], capture_output=True).returncode == 0


def run_in_container(command, timeout=120):
    args = [
        "docker", "run", "--rm",
        "--network", "none",
        "--read-only", "--tmpfs", "/tmp",
        "--memory", "256m", "--cpus", "1", "--pids-limit", "64",
        "--user", "1000:1000", "--cap-drop", "ALL", "--security-opt", "no-new-privileges",
        "-v", f"{workspace}:/work", "-w", "/work",
        IMAGE, "sh", "-c", command,
    ]
    done = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    return done.returncode, (done.stdout + done.stderr).strip()


network_test = "python -c \"import urllib.request; urllib.request.urlopen('https://example.com', timeout=5)\" && echo NETWORK WORKS || echo no network"
secret_test = f"cat {shlex.quote(str(elsewhere / 'secret.txt'))} >/dev/null 2>&1 && echo SECRET VISIBLE || echo secret not visible"

if docker_ready():
    subprocess.run(["docker", "pull", "-q", IMAGE], capture_output=True, timeout=300)
    _, network = run_in_container(network_test)
    _, secret = run_in_container(secret_test)
    print("Network inside the container:   ", network.splitlines()[-1])
    print("The secret file from section 2: ", secret.splitlines()[-1])
    assert network.splitlines()[-1] == "no network"
    assert secret.splitlines()[-1] == "secret not visible"
    print("Checkpoint passed.")
else:
    print("Docker isn't available here, so here's the command instead:\n")
    print("docker run --rm --network none --read-only --tmpfs /tmp \\")
    print("    --memory 256m --cpus 1 --pids-limit 64 --user 1000:1000 \\")
    print("    --cap-drop ALL --security-opt no-new-privileges \\")
    print("    -v \"$PWD:/work\" -w /work python:3.12-slim sh -c 'python -m unittest'")

Docker isn't available here, so here's the command instead:

docker run --rm --network none --read-only --tmpfs /tmp \
    --memory 256m --cpus 1 --pids-limit 64 --user 1000:1000 \
    --cap-drop ALL --security-opt no-new-privileges \
    -v "$PWD:/work" -w /work python:3.12-slim sh -c 'python -m unittest'


Containers still share the host's kernel, so a kernel bug can in principle let a process escape. For code from strangers on the internet, companies go further with gVisor-style kernels or lightweight virtual machines. For an agent working on your own project, a locked-down container is a very large step up from nothing, and most people run agents with nothing.

## 4. The sandboxed tool

For the agent we use the layers that work everywhere this notebook runs: workspace, minimal environment, process-group timeout, clipped output and resource limits where available. Swapping `run_limited` for `run_in_container` is a one-line change if you have Docker.

In [9]:
def run_command(command, timeout=20):
    command = re.sub(r"(?<![\w./-])python3?(?=\s|$)", lambda _: PYTHON, command)
    code, output, _ = run_limited(command, timeout=timeout, cpu_seconds=3)
    if len(output) > 4000:
        output = f"{output[:2000]}\n[... {len(output) - 4000:,} characters cut ...]\n{output[-2000:]}"
    return f"{describe(code)}\n{output}"


TOOLS = [{"type": "function", "function": {
    "name": "run_command",
    "description": "Run a shell command in the project folder, with limits on time, CPU and file size.",
    "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
}}]

And a script that never finishes. Whoever wrote it, model or human, the sandbox has the final say.

In [10]:
(workspace / "slow.py").write_text('print("starting", flush=True)\nwhile True:\n    pass\n')


def call_model(messages):
    payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()


def run_agent(task, max_steps=6):
    messages = [
        {"role": "system", "content": "You are a coding agent with a sandboxed terminal. Reply briefly."},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        message = call_model(messages)["choices"][0]["message"]
        messages.append(message)
        calls = message.get("tool_calls") or []
        if not calls:
            return {"status": "done", "answer": message.get("content")}
        for call in calls:
            try:
                command = json.loads(call["function"]["arguments"] or "{}").get("command", "")
                # This notebook has no permission layer, so keep the model to the one script we asked about.
                result = run_command(command) if command.strip() in ("python slow.py", "python3 slow.py") \
                    else "Denied: in this notebook the agent may only run `python slow.py`."
            except Exception as error:
                result = f"Error: {type(error).__name__}: {error}"
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            print(f"[{step}] $ {command}\n    {result.splitlines()[0]}")
    return {"status": "max_steps", "answer": None}


started = time.perf_counter()
run = run_agent("Run slow.py and tell me what it prints.")
print(f"\nAnswer: {run['answer']}")
print(f"Total time: {time.perf_counter() - started:.1f}s")

[1] $ python slow.py
    stopped: used more CPU time than allowed (SIGXCPU)



Answer: slow.py printed 'starting' and then never finished, so it was stopped. The tool reported: stopped: used more CPU time than allowed (SIGXCPU)
Total time: 4.2s


In [11]:
# Checkpoint: the runaway script was stopped and the agent still finished.
if llm.offline:
    assert run["status"] == "done" and "starting" in run["answer"]
print("Checkpoint passed.")

Checkpoint passed.


The agent didn't hang and didn't need special handling. The sandbox stopped the process, `describe` explained why, and the model passed that on. To an agent, a sandbox kill is just another tool result it can reason about.

## Recap

- Permissions judge command *text*. Sandboxes limit what running code can *do*. You want both.
- Resource limits stop runaway CPU, disk and memory use, but not file reads or network access.
- Containers add an isolated filesystem and can switch off the network entirely.
- Know which layer stops what. The table at the top of this notebook is worth remembering.

## Exercises

1. If you have Docker, switch `run_command` to `run_in_container` and rerun the agent. What changes in speed?
2. Add a limit on the number of processes (`RLIMIT_NPROC`). Why is it trickier than the others? (Hint: it counts every process your user is running.)
3. Make the container read-only for `/work` too, except for one `out/` folder. When would you want that?

## Clean up

In [12]:
shutil.rmtree(workspace)
shutil.rmtree(elsewhere)
print("Removed the temporary folders.")

Removed the temporary folders.


**Next:** [10. Agent v2](10_agent_v2.ipynb). Everything from Part 2 in one agent, on a harder task.